In [1]:
%matplotlib tk
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

/home/juan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
import matplotlib.pyplot as plt
import torch

from src.CocoDataset import CocoDataset
from src.modelo import crearModelo
from torchvision.transforms.functional import to_tensor
from src.campo import enCampo
from src.geometria import aMetros, puntoApoyo, calcularHomografia
from src.pipeline import analizarImagen
from src.calibracion import marcarPuntos
from src.viz import dibujarPuntosCenital, dibujarPuntos, dibujarCampo
from src.campo import separarPorCampo

ds = CocoDataset("../data/raw/football-players/valid")

modelo = crearModelo(congelarBackbone=False,ligero=False)
modelo.load_state_dict(torch.load("../outputs/modelo_experimento_ResNet_20ep.pt", map_location=torch.device('cpu')))
modelo.eval()

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [3]:
homografias = {}

In [5]:
IMG_ID = 5

ds = CocoDataset("../data/raw/football-players/test")

orden = ("medio_cercano", "area_izq_lejana", "medio_lejano", "penalti_izq")

fig, ax = plt.subplots()
correspondencias = marcarPuntos(ds.imagen(IMG_ID), orden)

H, Hinv = calcularHomografia(correspondencias)

with torch.no_grad():
    pred = modelo([to_tensor(ds.imagen(IMG_ID))])[0]

cajas = [
    b for b, l, s in zip(pred["boxes"].tolist(),
                         pred["labels"].tolist(),
                         pred["scores"].tolist())
    if int(l) == 3 and s >= 0.50
]

puntos = []

for caja in cajas:
    puntos.append(puntoApoyo(caja))

metros = aMetros(H  , puntos)

dentro, fuera = separarPorCampo(metros)

pares = analizarImagen(ds, IMG_ID, cajas)
equipos = [e for _, e in pares]

m0 = [m for m, e in zip(metros, equipos) if e == "e0" and enCampo(m)]
m1 = [m for m, e in zip(metros, equipos) if e == "e1" and enCampo(m)]

fig, ax = plt.subplots(figsize=(15, 10))
dibujarCampo(ax)
dibujarPuntosCenital(ax, m0)
dibujarPuntosCenital(ax, m1, color="red")
plt.show()